# 🎬 WanGP – Cena de teste AtivaMente Kids

**Antes de tudo:** menu *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4* → Salvar.

Depois: *Ambiente de execução → Executar tudo*. A primeira vez demora ~15–25 min (instala + baixa o modelo Wan 2.1 1.3B + gera o vídeo).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Instalar o WanGP (do nosso GitHub)

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/driveosescola3-afk/WanGP.git
%cd /content/WanGP
!pip install -q -r requirements.txt

## 2. Descrever a cena
Edite o `PROMPT` se quiser outra cena (o modelo entende melhor em inglês).

In [ ]:
PROMPT = (
    "A warm, cozy Brazilian home kitchen in soft morning light. A young mother and her "
    "5-year-old daughter sit together at a wooden table covered with colorful printed "
    "literacy worksheets with big letters, crayons and colored pencils. The little girl "
    "points at a large letter on the page and smiles proudly while the mother claps and "
    "smiles at her. Gentle slow camera push-in, shallow depth of field, realistic, "
    "natural skin tones, cheerful pastel colors, vertical video for social media."
)
NEGATIVE = "blurry, distorted face, extra fingers, deformed hands, text artifacts, watermark, dark, low quality"
RESOLUCAO = "480x832"   # vertical 9:16 (Reels/Stories). Use "832x480" para horizontal
FRAMES = 81             # ~5 segundos (16 fps)
PASSOS = 30             # menos passos = mais rápido, menos qualidade

## 3. Gerar o vídeo

In [ ]:
import os, sys
from pathlib import Path
os.chdir("/content/WanGP"); sys.path.insert(0, "/content/WanGP")
from shared.api import init

session = init(
    root=Path("/content/WanGP"),
    output_dir=Path("/content/saida"),
    cli_args=["--attention", "sdpa", "--profile", "4", "--fp16"],  # T4 não suporta bf16 bem
)

settings = session.get_default_settings("t2v_1.3B")
settings.update({
    "prompt": PROMPT,
    "negative_prompt": NEGATIVE,
    "resolution": RESOLUCAO,
    "video_length": FRAMES,
    "num_inference_steps": PASSOS,
    "seed": 42,
})

job = session.submit_task(settings)
for event in job.events.iter(timeout=1):
    if event.kind == "progress":
        p = event.data
        print(f"{p.phase} {p.current_step}/{p.total_steps}", end="\r")

result = job.result()
if result.success:
    VIDEO = result.generated_files[0]
    print("\n✅ Pronto:", VIDEO)
else:
    for e in result.errors: print("❌", e.message)

## 4. Ver e baixar

In [ ]:
from IPython.display import Video, display
from google.colab import files
display(Video(VIDEO, embed=True, width=360))
files.download(VIDEO)

---
### (Opcional) Abrir a interface completa do WanGP
Gera um link público temporário do Gradio para usar todos os modelos pela interface.

In [ ]:
%cd /content/WanGP
!python wgp.py --share --fp16 --attention sdpa --profile 4